In [8]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json
import numpy as np


# Function to load idx2word and convert it to word2idx
def load_vocabulary(path):
    with open(path, 'r') as file:
        idx2word = json.load(file)
    word2idx = {v: int(k) for k, v in idx2word.items()}
    return idx2word, word2idx

# Load vocabulary
idx2word_path = '/home/vitoupro/code/image_captioning/notebook/idx2word.json'
idx2word, word2idx = load_vocabulary(idx2word_path)

# Model Definitions (EncoderCNN and DecoderRNN)
class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for name, param in resnet.named_parameters():
            param.requires_grad = 'layer4' in name
        modules = list(resnet.children())[:-2]  # Keep conv feature map
        self.resnet = nn.Sequential(*modules)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((14, 14))
        self.embed = nn.Linear(2048, embed_size)

    def forward(self, images):
        features = self.resnet(images)  # (B, 2048, H, W)
        features = self.adaptive_pool(features)  # (B, 2048, 14, 14)
        features = features.mean(dim=[2, 3])  # (B, 2048) - Global average pooling
        features = self.embed(features)  # (B, embed_size)
        return features

class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1, dropout_prob=0.3):
        super(DecoderRNN, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(dropout_prob)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.init_h = nn.Linear(512, hidden_size)  # Initialize from encoder output (512)
        self.init_c = nn.Linear(512, hidden_size)  # Initialize from encoder output (512)

    def forward(self, features, captions):
        embeddings = self.embed(captions)
        embeddings = self.dropout(embeddings)
        h0 = self.init_h(features).unsqueeze(0).repeat(self.num_layers, 1, 1)
        c0 = self.init_c(features).unsqueeze(0).repeat(self.num_layers, 1, 1)
        lstm_out, _ = self.lstm(embeddings, (h0, c0))
        lstm_out = self.dropout(lstm_out)
        outputs = self.linear(lstm_out)
        return outputs

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize models with the correct vocab size (use actual vocab size from loaded vocabulary)
print(f"Vocabulary size: {len(word2idx)}")
encoder = EncoderCNN(embed_size=512).to(device)
decoder = DecoderRNN(embed_size=256, hidden_size=512, vocab_size=len(word2idx), num_layers=1, dropout_prob=0.3).to(device)

# Load model weights
encoder.load_state_dict(torch.load('encoder_experiment16_best.pth'))
decoder.load_state_dict(torch.load('decoder_experiment16_best.pth'))
encoder.eval()
decoder.eval()

# Function to generate a caption using greedy decoding
def generate_caption(image_path, encoder, decoder, idx2word, word2idx, transform, max_length=30):
    image = Image.open(image_path).convert("RGB")
    if transform:
        image = transform(image)
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        features = encoder(image)
        input_seq = torch.tensor([[word2idx['<START>']]]).to(device)
        
        caption_ids = []
        
        for _ in range(max_length):
            outputs = decoder(features, input_seq)
            predicted_id = outputs.argmax(-1)[:, -1].item()
            
            if predicted_id == word2idx['<END>']:
                break
                
            caption_ids.append(predicted_id)
            
            # Append predicted id to input sequence
            input_seq = torch.cat([input_seq, torch.tensor([[predicted_id]]).to(device)], dim=1)
        
        # Convert ids to text, only use ids that exist in vocabulary
        caption_text = ""
        for idx in caption_ids:
            if str(idx) in idx2word:
                caption_text += idx2word[str(idx)]
        
        return caption_text

# Transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

# Image path
image_path = '/home/vitoupro/code/image_captioning/data/00000001_020.jpg'

caption = generate_caption(image_path, encoder, decoder, idx2word, word2idx, transform)
print("Generated Caption:", caption)

Vocabulary size: 70
Generated Caption: សត្រលើថ្រលើថ្រលើថ្រលើថ្រលើថ្រល
